In [ ]:
ENV["CLIMACOMMS_DEVICE"] = "CUDA"
ENV["CLIMACOMMS_CONTEXT"] = "SINGLETON"
# ENV["JULIA_MPI_HAS_CUDA"] = true

"MPI"

In [2]:
import ClimaComms
ClimaComms.@import_required_backends
using ClimaUtilities.ClimaArtifacts
import ClimaUtilities.TimeManager: ITime, date

import ClimaDiagnostics
import ClimaUtilities

import ClimaUtilities.TimeVaryingInputs:
    TimeVaryingInput, LinearInterpolation, PeriodicCalendar
import ClimaUtilities.ClimaArtifacts: @clima_artifact
import ClimaParams as CP
using ClimaCore
using ClimaLand
using ClimaLand.Snow
using ClimaLand.Soil
using ClimaLand.Canopy
import ClimaLand
import ClimaLand.Parameters as LP
import ClimaLand.Simulations: LandSimulation, solve!

using Dates

using CairoMakie, GeoMakie, ClimaAnalysis
import ClimaLand.LandSimVis as LandSimVis

In [3]:
const FT = Float64;
context = ClimaComms.context()
ClimaComms.init(context)
device = ClimaComms.device()
device_suffix = device isa ClimaComms.CPUSingleThreaded ? "cpu" : "gpu"
root_path = "land_longrun_$(device_suffix)"
diagnostics_outdir = joinpath(root_path, "global_diagnostics")
outdir =
    ClimaUtilities.OutputPathGenerator.generate_output_path(diagnostics_outdir);
toml_dict = LP.create_toml_dict(FT)

LoadError: MPI implementation is not built with CUDA-aware interface. If your MPI is not OpenMPI, you have to set JULIA_MPI_HAS_CUDA to `true`

In [ ]:
parameter_log_file = joinpath(root_path, "parameters.toml")
CP.log_parameter_information(toml_dict, parameter_log_file)

┌ Warning: Keys are present in parameter file but not used in the simulation. 
│  Typically this is due to a mismatch in parameter name in toml and in source. Offending keys: Any["jordan_quadratic_snow_thermal_conductivity", "albedo_psnow_reset_rate", "slab_lake_emissivity", "beta_min", "jordan_linear_snow_thermal_conductivity", "canopy_emissivity", "bucket_z_0m", "leaf_Cd", "O2_diffusion_coefficient_air", "relative_contribution_factor", "optimal_lai_tau_long_term", "pmodel_Ha_Vcmax", "kf", "f_over", "moisture_stress_pc", "pmodel_ϕ0_c3", "sturm_m1_snow_thermal_conductivity", "molar_mass_oxygen", "autotrophic_respiration_T_ref", "kd_p2", "z0", "pmodel_ϕ0_c4", "pmodel_ϕa1_c4", "sturm_threshold_snow_thermal_conductivity", "pmodel_bRd", "canopy_K_lw", "bucket_z_0b", "canopy_height", "critical_snow_fraction", "soil_C_substrate_diffusivity", "michaelis_constant", "molar_mass_carbon", "canopy_z_0min", "O2_henry_k298", "plant_S_s", "kn_p2", "slab_lake_z_0b", "autotrophic_respiration_Q10", "RAI

In [ ]:
Δt = 450.0
start_date = DateTime(2008)
stop_date = DateTime(2009);

nelements = (18, 36, 7)
domain = ClimaLand.Domains.global_box_domain(FT; context, nelements);

In [ ]:
forcing = ClimaLand.prescribed_forcing_era5(
    start_date,
    stop_date,
    domain.space.surface,
    toml_dict,
    FT;
    use_lowres_forcing = true,
    max_wind_speed = 25.0,
    time_interpolation_method = LinearInterpolation(PeriodicCalendar()),
    regridder_type = :InterpolationsRegridder,
    context,
);

LAI = ClimaLand.Canopy.prescribed_lai_modis(
    domain.space.surface,
    start_date,
    stop_date,
);

: 

In [ ]:
model = ClimaLand.LandModel{FT}(forcing, LAI, toml_dict, domain, Δt);
simulation = ClimaLand.Simulations.LandSimulation(
    start_date,
    stop_date,
    Δt,
    model;
    outdir,
    user_callbacks = (),
);

In [ ]:
ClimaLand.Simulations.solve!(simulation)